Generate frames for annotation

Imports

In [4]:
import ffmpeg
import os
import random
import subprocess
import numpy as np

Looping through every video stored in VODS, scrpit extracts 100 frames with a normal distribrution to be more efficient with getting the "meat" from the middle instead of caster segments + intro. Frames stores in "Frames" and list of videos already sampled stored in "sampled_videos.txt".

In [5]:
# Define input folder containing videos
video_folder = "VODS"
log_file = "sampled_videos.txt"
output_folder = "Frames"

# Ensure output folder exists
os.makedirs(output_folder, exist_ok=True)

# Load processed videos from log file
if os.path.exists(log_file):
    with open(log_file, "r") as f:
        processed_videos = set(f.read().splitlines())
else:
    processed_videos = set()

# Get list of videos in the folder
video_files = [f for f in os.listdir(video_folder) if f.endswith((".mp4", ".webm"))]

for video in video_files:
    video_path = os.path.join(video_folder, video)

    # Skip if already processed
    if video in processed_videos:
        print(f"Skipping {video}, already processed.")
        continue

    # Step 1: Get video duration
    cmd = f'ffprobe -i "{video_path}" -show_entries format=duration -v quiet -of csv="p=0"'
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    try:
        video_length = float(result.stdout.strip())
    except ValueError:
        print(f"Could not determine duration for {video}, skipping.")
        continue

    # Step 2: Generate 100 timestamps using normal distribution
    mu = video_length / 2  # Mean at the middle
    sigma = video_length / 4  # Adjust std to keep values within range

    timestamps = np.random.normal(mu, sigma, 100)
    timestamps = np.clip(timestamps, 0, video_length)  # Keep within video bounds
    timestamps = np.sort(timestamps)  # Sort to maintain chronological order

    # Step 3: Extract frames at these timestamps using ffmpeg
    video_name = os.path.splitext(video)[0]  # Remove extension
    video_output_folder = os.path.join(output_folder, video_name)
    os.makedirs(video_output_folder, exist_ok=True)

    for i, timestamp in enumerate(timestamps):
        output_image = os.path.join(video_output_folder, f"frame_{i:03d}.png")
        cmd = f'ffmpeg -ss {timestamp:.2f} -i "{video_path}" -frames:v 1 "{output_image}" -y -hide_banner -loglevel error'
        subprocess.run(cmd, shell=True)

    # Step 4: Log the processed video
    with open(log_file, "a") as f:
        f.write(video + "\n")

    print(f"Processed {video} successfully!")

print("All videos processed.")


Processed G2 vs. SEN  - VCT Americas Kickoff - Day 12 - Map 04.webm successfully!
Skipping G2 vs. SEN  - VCT Americas Kickoff - Day 12 - Map 05.mp4, already processed.
All videos processed.


In [7]:
print(video_files)

['G2 vs. SEN  - VCT Americas Kickoff - Day 12 - Map 04.webm', 'G2 vs. SEN  - VCT Americas Kickoff - Day 12 - Map 05.mp4']
